# 03 - Signals and trades

What the strategies see: per-bar multi-horizon consensus, confidence and variance spikes,
next to price and the trades they produced. Uses the same out-of-sample TEST block as 02.

In [ ]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
STRATEGY = "liberal"
WINDOW = 1500                   # bars to plot (the last WINDOW of the test block)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from neural_trade.data.processor import split_arrays
from neural_trade.serving.predictor import Predictor
from neural_trade.strategy import BacktestConfig, Bars, SignalFrame, build_strategy, run_backtest

run_dir = Path(RUN_DIR) if RUN_DIR else max((p for p in Path(RUNS_DIR).iterdir() if (p / "artifacts").is_dir()), key=os.path.getmtime)
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy().override(CSV_PATH=CSV_PATH)
blocks = split_arrays(cfg)
test = blocks["test"]
frame = predictor.predict(test["X"], test["last_close"]).to_prediction_frame(
    predictor.bundle.pred_scale, predictor.bundle.pred_mean, y=test["y"], split="test")
bars = Bars.from_frame(blocks["df"], test["anchor_bar"])
s = SignalFrame.build(frame, predictor.bundle.meta["var_scale"])
res = run_backtest(s, bars, build_strategy(STRATEGY), BacktestConfig())

## Signal summary

In [ ]:
features = pd.DataFrame({"weighted_direction": s.weighted_direction, "strength": s.strength,
                         "avg_confidence": s.avg_confidence, "agreement": s.agreement, "consensus": s.consensus,
                         "magnitude_coherent": s.magnitude_coherent, "direction_aligned": s.direction_aligned,
                         "var_spike": s.var_spike, "volatility_$": s.volatility})
features.describe().T

In [ ]:
lo = max(0, len(s) - WINDOW)
x = np.arange(lo, len(s))
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[0.4, 0.2, 0.2, 0.2], vertical_spacing=0.03,
                    subplot_titles=("Close and trades", "Weighted P(up)", "Strength and confidence", "h1 variance"))
fig.add_trace(go.Scatter(x=x, y=s.close[lo:], name="close", line=dict(width=1)), 1, 1)
for t in [t for t in res.trades if t.entry_bar >= lo]:
    fig.add_trace(go.Scatter(x=[t.entry_bar, t.exit_bar], y=[t.entry_price, t.exit_price], mode="lines+markers",
                             line=dict(color="#15803d" if t.net_pnl > 0 else "#b91c1c", width=2),
                             showlegend=False, hovertext=f"{t.side} {t.exit_reason} {t.net_pnl:+.2f}"), 1, 1)
fig.add_trace(go.Scatter(x=x, y=s.weighted_direction[lo:], name="weighted P(up)"), 2, 1)
fig.add_hline(y=0.5, line_dash="dot", row=2, col=1)
fig.add_trace(go.Scatter(x=x, y=s.strength[lo:], name="strength"), 3, 1)
fig.add_trace(go.Scatter(x=x, y=s.avg_confidence[lo:], name="confidence"), 3, 1)
fig.add_trace(go.Scatter(x=x, y=s.var_scaled[lo:, 1], name="var h1"), 4, 1)
fig.add_trace(go.Scatter(x=x[s.var_spike[lo:]], y=s.var_scaled[lo:, 1][s.var_spike[lo:]], mode="markers",
                         name="spike", marker=dict(color="#b45309", size=5)), 4, 1)
fig.update_layout(height=900, hovermode="x unified")
fig.show()

## Trades by exit reason

In [ ]:
trades = res.trades_frame()
trades.groupby("exit_reason")[["net_pnl", "bars_held"]].agg(["count", "mean", "sum"]) if len(trades) else trades